# AIU重点テーマ分析（2_02）

保存済みLLM分類と元のtrain/testを結合し、`admin_process × ai_usecase`を1テーマとして可視化・順位付けします。LLM APIは呼びません。

分析対象は`project_start_year >= 2020`です。`ai_usecase`はexplodeし、複数ラベルの事業は最大3テーマへ1回ずつ寄与します。`U99`とHigh-app 0件のテーマは診断・集計には残しますが、重点テーマランキングから除外します。

In [ ]:
from pathlib import Path
import sys

import japanize_matplotlib  # noqa: F401
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from llm_features import ADMIN_PROCESSES, AI_USECASES, POLICY_DOMAINS, decode_ai_usecase
from market_analysis import (
    DEFAULT_ADMIN_FIT,
    DEFAULT_USECASE_FIT,
    DEFAULT_WEIGHT_PROFILES,
    STRATEGIC_WEIGHTS,
    build_theme_long,
    compare_weight_profiles,
    compute_theme_metrics,
    compute_yearly_theme_metrics,
    merge_project_taxonomy,
    score_theme_metrics,
)

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 140)

def save_figure(name: str) -> None:
    path = OUTPUT_DIR / name
    plt.savefig(path, dpi=180, bbox_inches='tight')
    print('saved:', path)

## 1. パスと分析設定

AIU Fitと総合スコアはLLMの主観評価ではなく、下記の明示的な辞書と重みから決まります。組織内の合意に合わせて、このセルだけを編集してください。

In [ ]:
TRAIN_PATH = PROJECT_ROOT / 'input' / 'train.csv'
TEST_PATH = PROJECT_ROOT / 'input' / 'test.csv'
LLM_FEATURE_PATH = PROJECT_ROOT / 'data' / 'csv' / 'ai_market_llm_features.csv.gz'
METRICS_PATH = PROJECT_ROOT / 'data' / 'csv' / 'ai_market_theme_metrics.csv'
YEARLY_METRICS_PATH = PROJECT_ROOT / 'data' / 'csv' / 'ai_market_theme_yearly_metrics.csv'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'ai_market_themes'

ID_COL = 'project_id'
YEAR_COL = 'project_start_year'
MINISTRY_COL = 'responsible_ministry'
MIN_YEAR = 2020
PRIOR_STRENGTH = 20.0
TOP_N = 15
TOP_TREND_N = 8
SELECTED_THEME_CODE = None  # 例: 'A03__U05'。Noneなら総合1位

ADMIN_FIT = DEFAULT_ADMIN_FIT.copy()
USECASE_FIT = DEFAULT_USECASE_FIT.copy()
WEIGHTS = STRATEGIC_WEIGHTS.copy()
WEIGHT_PROFILES = {name: values.copy() for name, values in DEFAULT_WEIGHT_PROFILES.items()}

display(pd.DataFrame({
    'admin_process': list(ADMIN_FIT),
    'label': [ADMIN_PROCESSES[code] for code in ADMIN_FIT],
    'fit': list(ADMIN_FIT.values()),
}))
display(pd.DataFrame({
    'ai_usecase': list(USECASE_FIT),
    'label': [AI_USECASES[code] for code in USECASE_FIT],
    'fit': list(USECASE_FIT.values()),
}))
display(pd.Series(WEIGHTS, name='weight').to_frame())

## 2. 元データとLLM分類結果を安全に結合

`source_split + project_id`を一意キーとしてone-to-one結合します。対象年度の元データとLLM結果が1件でも不一致なら停止するため、不完全な分類結果のままランキングを作りません。

In [ ]:
required_files = [TRAIN_PATH, TEST_PATH, LLM_FEATURE_PATH]
missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        '必要なファイルがありません。Notebook 2_01でLLM特徴量を生成してから実行してください: '
        + ', '.join(str(path) for path in missing_files)
    )

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
llm_features = pd.read_csv(LLM_FEATURE_PATH, compression='gzip')

projects, merge_diagnostics = merge_project_taxonomy(
    train,
    test,
    llm_features,
    id_col=ID_COL,
    year_col=YEAR_COL,
    ministry_col=MINISTRY_COL,
    min_year=MIN_YEAR,
)
display(merge_diagnostics.to_frame())
display(projects.head())
assert projects['project_key'].is_unique
assert projects[YEAR_COL].ge(MIN_YEAR).all()

## 3. 分類品質と分析母集団の診断

省庁欠損はbreadth・HHIから除外し、欠損率を別表示します。U99は全体品質の診断には含めます。

In [ ]:
theme_long = build_theme_long(projects)
u99_by_project = projects['ai_usecase'].map(decode_ai_usecase).map(lambda values: values == ['U99'])
diagnostics = pd.Series({
    'project_rows': len(projects),
    'theme_assignment_rows': len(theme_long),
    'observed_themes_including_u99': theme_long['theme_code'].nunique(),
    'multi_usecase_project_rate': projects['ai_usecase'].map(decode_ai_usecase).map(len).gt(1).mean(),
    'U99_project_rate': u99_by_project.mean(),
    'high_app_project_rate': projects['ai_applicability'].eq(2).mean(),
    'ministry_missing_rate': projects['responsible_ministry'].isna().mean(),
    'minimum_year': projects[YEAR_COL].min(),
    'maximum_year': projects[YEAR_COL].max(),
})
display(diagnostics.to_frame('value'))
display(pd.crosstab(projects['source_split'], projects['ai_applicability'], margins=True))
display(pd.crosstab(theme_long['admin_process'], theme_long['ai_usecase']))

## 4. 6指標・総合順位・年度別指標を作成

High-appは`ai_applicability == 2`だけです。High-app rateの表示値は生の比率、順位には全テーマのHigh-app率を事前値・強度20件とした平滑化率を使います。HHIは高いほど集中しており、総合順位では`1 - HHI`を評価します。

In [ ]:
theme_metrics = compute_theme_metrics(
    theme_long,
    prior_strength=PRIOR_STRENGTH,
    admin_fit=ADMIN_FIT,
    usecase_fit=USECASE_FIT,
)
scored_metrics = score_theme_metrics(theme_metrics, weights=WEIGHTS)
yearly_metrics = compute_yearly_theme_metrics(theme_long)
profile_comparison = compare_weight_profiles(theme_metrics, profiles=WEIGHT_PROFILES)

profile_columns = ['theme_code'] + [
    column for column in profile_comparison.columns if column.endswith('_score') or column.endswith('_rank')
]
metrics_output = scored_metrics.merge(
    profile_comparison[profile_columns], on='theme_code', how='left', validate='one_to_one'
)

METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
metrics_output.to_csv(METRICS_PATH, index=False)
yearly_metrics.to_csv(YEARLY_METRICS_PATH, index=False)
top_themes = metrics_output.loc[metrics_output['ranking_eligible']].head(TOP_N).copy()
top_themes.to_csv(OUTPUT_DIR / 'top_themes.csv', index=False)
profile_comparison.to_csv(OUTPUT_DIR / 'weight_profile_comparison.csv', index=False)
print('saved:', METRICS_PATH)
print('saved:', YEARLY_METRICS_PATH)
print('saved:', OUTPUT_DIR / 'top_themes.csv')

In [ ]:
ranking_columns = [
    'overall_rank', 'theme_label', 'project_count', 'high_app_project_count',
    'high_app_rate', 'high_app_rate_smoothed', 'ministry_breadth', 'domain_breadth',
    'ministry_hhi', 'domain_hhi', 'concentration_hhi', 'aiu_fit', 'overall_score',
]
display(top_themes[ranking_columns].style.format({
    'high_app_rate': '{:.1%}',
    'high_app_rate_smoothed': '{:.1%}',
    'ministry_hhi': '{:.3f}',
    'domain_hhi': '{:.3f}',
    'concentration_hhi': '{:.3f}',
    'overall_score': '{:.1f}',
}).background_gradient(subset=['overall_score'], cmap='YlGn'))

## 5. admin_process × ai_usecase ヒートマップ

観測されたテーマだけを着色します。Concentrationは赤いほど特定省庁・分野への偏りが強く、他の指標とは評価方向が逆です。

In [ ]:
heatmap_specs = [
    ('high_app_project_count', '① High-app project count', 'Blues', None, None),
    ('high_app_rate', '② High-app rate', 'YlGn', 0, 1),
    ('ministry_breadth', '③ Ministry breadth', 'PuBu', None, None),
    ('domain_breadth', '④ Domain breadth', 'BuPu', None, None),
    ('concentration_hhi', '⑤ Concentration HHI（低いほど分散）', 'Reds', 0, 1),
    ('aiu_fit', '⑥ AIU Fit', 'viridis', 0, 5),
    ('overall_score', '総合スコア', 'YlOrRd', 0, 100),
]
admin_order = list(ADMIN_PROCESSES)
usecase_order = [code for code in AI_USECASES if code != 'U99']
fig, axes = plt.subplots(4, 2, figsize=(22, 30))
for ax, (column, title, cmap, vmin, vmax) in zip(axes.flat, heatmap_specs):
    pivot = metrics_output.loc[metrics_output['ai_usecase'].ne('U99')].pivot(
        index='admin_process', columns='ai_usecase', values=column
    ).reindex(index=admin_order, columns=usecase_order)
    sns.heatmap(
        pivot, mask=pivot.isna(), cmap=cmap, vmin=vmin, vmax=vmax,
        linewidths=0.4, linecolor='white', ax=ax, cbar_kws={'shrink': 0.75}
    )
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('ai_usecase')
    ax.set_ylabel('admin_process')
axes.flat[-1].axis('off')
plt.tight_layout()
save_figure('theme_metric_heatmaps.png')
plt.show()

## 6. 市場量とHigh-app率のバブルチャート

右上ほど案件数とHigh-app率がともに高いテーマです。バブルサイズはHigh-app事業の省庁数、色はAIU Fitを表します。

In [ ]:
eligible = metrics_output.loc[metrics_output['ranking_eligible']].copy()
fig, ax = plt.subplots(figsize=(15, 9))
scatter = ax.scatter(
    eligible['high_app_rate'],
    eligible['high_app_project_count'],
    s=100 + eligible['ministry_breadth'] * 80,
    c=eligible['aiu_fit'],
    cmap='viridis', vmin=0, vmax=5, alpha=0.75, edgecolor='black', linewidth=0.5,
)
for row in eligible.nsmallest(min(TOP_N, len(eligible)), 'overall_rank').itertuples():
    ax.annotate(
        row.theme_code, (row.high_app_rate, row.high_app_project_count),
        xytext=(5, 5), textcoords='offset points', fontsize=9
    )
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel('High-app rate')
ax.set_ylabel('High-app project count')
ax.set_title('重点テーマ候補：案件数 × High-app率')
colorbar = plt.colorbar(scatter, ax=ax)
colorbar.set_label('AIU Fit (0–5)')
plt.tight_layout()
save_figure('theme_bubble_count_rate.png')
plt.show()

## 7. 上位テーマのスコア構成と指標プロファイル

総合点だけでなく、どの指標が順位を押し上げているかを確認します。

In [ ]:
component_labels = {
    'high_app_project_count': 'High-app件数',
    'high_app_rate_smoothed': 'High-app率（平滑化）',
    'ministry_breadth': '省庁breadth',
    'domain_breadth': '分野breadth',
    'low_concentration': '低集中度',
    'aiu_fit': 'AIU Fit',
}
top_plot = eligible.nsmallest(min(TOP_N, len(eligible)), 'overall_rank').copy()
contributions = pd.DataFrame(index=top_plot['theme_code'])
for component, weight in WEIGHTS.items():
    contributions[component_labels[component]] = (
        top_plot.set_index('theme_code')[f'{component}_score'] * weight
    )
ax = contributions.iloc[::-1].plot.barh(stacked=True, figsize=(14, 9), colormap='tab20c')
ax.set_xlabel('総合スコアへの寄与')
ax.set_ylabel('theme')
ax.set_title('上位テーマの総合スコア構成')
ax.legend(loc='lower right')
plt.tight_layout()
save_figure('top_theme_score_contributions.png')
plt.show()

profile_columns = [f'{component}_score' for component in WEIGHTS]
profile = top_plot.set_index('theme_code')[profile_columns].rename(
    columns={f'{key}_score': value for key, value in component_labels.items()}
)
plt.figure(figsize=(12, max(6, len(profile) * 0.55)))
sns.heatmap(profile, cmap='YlGnBu', vmin=0, vmax=100, annot=True, fmt='.0f')
plt.title('上位テーマの指標プロファイル（percentile score）')
plt.xlabel('')
plt.ylabel('theme')
plt.tight_layout()
save_figure('top_theme_metric_profiles.png')
plt.show()

## 8. 上位テーマの省庁・policy domain構成

High-app事業だけを使い、横展開の広さと特定領域への偏りを構成比で確認します。欠損省庁は除外します。

In [ ]:
top_codes = top_plot['theme_code'].head(TOP_TREND_N).tolist()
high_top = theme_long.loc[
    theme_long['is_high_app'] & theme_long['theme_code'].isin(top_codes)
].copy()
theme_order = [code for code in top_codes if code in set(high_top['theme_code'])]

ministry_counts = pd.crosstab(high_top['theme_code'], high_top['responsible_ministry']).reindex(theme_order)
if not ministry_counts.empty:
    top_ministries = ministry_counts.sum().nlargest(15).index
    ministry_share = ministry_counts[top_ministries].div(ministry_counts.sum(axis=1), axis=0)
    plt.figure(figsize=(16, max(6, len(ministry_share) * 0.65)))
    sns.heatmap(ministry_share, cmap='Blues', vmin=0, vmax=1, annot=True, fmt='.0%')
    plt.title('上位テーマの省庁構成比（High-app事業、上位15省庁）')
    plt.xlabel('responsible_ministry')
    plt.ylabel('theme')
    plt.tight_layout()
    save_figure('top_theme_ministry_mix.png')
    plt.show()
else:
    print('省庁がすべて欠損しているため、省庁構成図を作成できません。')

domain_counts = pd.crosstab(high_top['theme_code'], high_top['policy_domain']).reindex(
    index=theme_order, columns=list(POLICY_DOMAINS), fill_value=0
)
domain_share = domain_counts.div(domain_counts.sum(axis=1), axis=0)
plt.figure(figsize=(16, max(6, len(domain_share) * 0.65)))
sns.heatmap(domain_share, cmap='Purples', vmin=0, vmax=1, annot=True, fmt='.0%')
plt.title('上位テーマのpolicy domain構成比（High-app事業）')
plt.xlabel('policy_domain')
plt.ylabel('theme')
plt.tight_layout()
save_figure('top_theme_domain_mix.png')
plt.show()

## 9. 年度推移

上位テーマが特定年度だけの現象ではないかを、High-app件数と率の両方で確認します。データがない年度の件数は0、率は欠損として扱います。

In [ ]:
years = list(range(int(projects[YEAR_COL].min()), int(projects[YEAR_COL].max()) + 1))
trend_grid = pd.MultiIndex.from_product(
    [years, top_codes], names=[YEAR_COL, 'theme_code']
).to_frame(index=False)
trend = trend_grid.merge(
    yearly_metrics.loc[yearly_metrics['theme_code'].isin(top_codes)],
    on=[YEAR_COL, 'theme_code'], how='left'
)
trend['project_count'] = trend['project_count'].fillna(0).astype(int)
trend['high_app_project_count'] = trend['high_app_project_count'].fillna(0).astype(int)
trend['high_app_rate'] = trend['high_app_project_count'].div(
    trend['project_count'].replace(0, np.nan)
)

fig, axes = plt.subplots(2, 1, figsize=(15, 13), sharex=True)
sns.lineplot(
    data=trend, x=YEAR_COL, y='high_app_project_count', hue='theme_code',
    marker='o', ax=axes[0]
)
sns.lineplot(
    data=trend, x=YEAR_COL, y='high_app_rate', hue='theme_code',
    marker='o', ax=axes[1], legend=False
)
axes[0].set_title('上位テーマの年度別High-app project count')
axes[1].set_title('上位テーマの年度別High-app rate')
axes[1].yaxis.set_major_formatter(PercentFormatter(1.0))
axes[0].legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
save_figure('top_theme_yearly_trends.png')
plt.show()

## 10. 重み感度

戦略重視・均等・市場規模重視の3設定で順位を比較します。どの設定でも上位に残るテーマほど、重みへの依存が小さい候補です。

In [ ]:
rank_columns = [f'{name}_rank' for name in WEIGHT_PROFILES]
sensitivity = profile_comparison.loc[profile_comparison['ranking_eligible']].copy()
sensitivity['best_rank'] = sensitivity[rank_columns].min(axis=1)
sensitivity['worst_rank'] = sensitivity[rank_columns].max(axis=1)
sensitivity['rank_spread'] = sensitivity['worst_rank'] - sensitivity['best_rank']
sensitivity_top = sensitivity.loc[sensitivity['best_rank'].le(TOP_N)].sort_values(
    ['strategic_rank', 'rank_spread']
)
display(sensitivity_top[[
    'theme_label', *rank_columns, 'best_rank', 'worst_rank', 'rank_spread'
]])
plt.figure(figsize=(10, max(6, len(sensitivity_top) * 0.5)))
sns.heatmap(
    sensitivity_top.set_index('theme_code')[rank_columns],
    cmap='YlGn_r', annot=True, fmt='.0f', linewidths=0.5
)
plt.title('重み設定別の順位（小さいほど上位）')
plt.xlabel('weight profile')
plt.ylabel('theme')
plt.tight_layout()
save_figure('weight_sensitivity_ranks.png')
plt.show()

## 11. 選択テーマを構成する事業の確認

総合順位は最終判断ではありません。上位テーマを選び、元事業・分類理由・省庁・分野を確認して、分類品質と事業内容の一貫性を人間がレビューします。

In [ ]:
if eligible.empty:
    raise RuntimeError('High-app事業を含むランキング対象テーマがありません。')
selected_theme = SELECTED_THEME_CODE or eligible.nsmallest(1, 'overall_rank')['theme_code'].iloc[0]
if selected_theme not in set(eligible['theme_code']):
    raise ValueError(f'SELECTED_THEME_CODE is not ranking eligible: {selected_theme}')

display(metrics_output.loc[metrics_output['theme_code'].eq(selected_theme), ranking_columns])
selected_projects = theme_long.loc[theme_long['theme_code'].eq(selected_theme)].copy()
review_columns = [
    column for column in [
        'source_split', ID_COL, YEAR_COL, 'project_name', 'responsible_ministry',
        'policy_domain', 'ai_applicability', 'classification_reason'
    ] if column in selected_projects.columns
]
display(selected_projects.sort_values(
    ['ai_applicability', YEAR_COL], ascending=[False, False]
)[review_columns].head(50))
display(selected_projects.loc[selected_projects['is_high_app'], 'responsible_ministry'].value_counts(dropna=False).to_frame('high_app_projects'))
display(selected_projects.loc[selected_projects['is_high_app'], 'policy_domain'].value_counts().rename(index=POLICY_DOMAINS).to_frame('high_app_projects'))

## 判断時の注意

- 総合順位は仮説形成の補助であり、AIU Fit辞書と重みの合意形成が必要です。
- 複数usecaseの事業は複数テーマへ寄与するため、テーマ間の件数は排他的ではありません。
- High-app rateの生値と平滑化値を併記し、少数テーマの100%を過大評価しないでください。
- 年度推移、元事業、classification_reasonを確認してから重点テーマを確定してください。
- 予算規模は今回の総合スコアに含めていません。必要なら次の拡張として追加します。